# Simulação em vídeo

Este notebook aplica o modelo **YOLOv11** treinado do Urban Disaster Monitor a um vídeo, gerando um novo vídeo com as detecções em bounding boxes (civis, socorristas, animais). Útil para testar o modelo em cenários dinâmicos, como o [vídeo de bombeiros em área inundada](https://www.youtube.com/watch?v=QnFwDqzCwRU) usado no projeto.

---
**Projeto:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Autores:** Carolina Soares, João Galdino

## 0. Dependências

Instale Ultralytics e OpenCV (e tqdm para barra de progresso). No Colab, pode ser necessário ativar GPU em *Runtime → Change runtime type*.

In [ ]:
!pip install ultralytics opencv-python tqdm --quiet

## 1. Configuração

Defina os caminhos do **modelo** (pesos treinados), do **vídeo de entrada** e do **vídeo de saída**. Ajuste `CONFIDENCE` para filtrar detecções (ex.: 0.75 para maior precisão).

In [ ]:
from pathlib import Path

MODEL_PATH = "../models/yolov11n/best.pt"
VIDEO_INPUT = "../static/video/rescuer.mp4"
VIDEO_OUTPUT = "saida_detectada.mp4"
CONFIDENCE = 0.75  # confiança mínima para detecção

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f"Modelo não encontrado: {MODEL_PATH}. Treine o modelo ou ajuste MODEL_PATH.")
if not Path(VIDEO_INPUT).exists():
    raise FileNotFoundError(f"Vídeo não encontrado: {VIDEO_INPUT}. Coloque o vídeo no projeto ou ajuste VIDEO_INPUT.")
print("Configuração OK.")

## 2. Processar vídeo

Carrega o modelo, lê o vídeo frame a frame, aplica a detecção e grava o resultado. A barra de progresso indica o andamento.

In [ ]:
import cv2
from ultralytics import YOLO

try:
    from tqdm import tqdm
    use_tqdm = True
except ImportError:
    use_tqdm = False

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_INPUT)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (width, height))

pbar = tqdm(total=total_frames, desc="Processando", unit="frame") if use_tqdm else None
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=CONFIDENCE, verbose=False)
    result_img = results[0].plot()
    out.write(result_img)
    frame_idx += 1
    if pbar:
        pbar.update(1)
    elif frame_idx % 30 == 0:
        print(f"Frames processados: {frame_idx}/{total_frames}")
if pbar:
    pbar.close()
cap.release()
out.release()
print(f"Vídeo salvo: {VIDEO_OUTPUT}")

## 3. Reproduzir o vídeo de saída

In [ ]:
from IPython.display import Video
Video(VIDEO_OUTPUT, embed=True, width=640)